# Persistent Colab workspace

Keep files and a saved Python environment in Google Drive, then restore them on a new CPU or GPU runtime.

**Start:** choose CPU for ordinary development, or select a GPU when needed. Run cells 1 and 2 before importing your own packages. Grant Drive access if Google asks. Use the same project name on future visits.

**Finish:** save model/training checkpoints, run **Save environment**, then **Flush Drive**. Finally use **Runtime → Disconnect and delete runtime** to release compute.

Drive preserves files and saved environment archives. A released VM loses RAM, GPU memory, running programs and unsaved changes. A Python/base-image change requires an explicit environment rebuild. Google Drive capacity does not enlarge the runtime disk.


In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. Restore workspace and Python environment
PROJECT = 'default'  # @param {type:"string"}
exec('"""Restore a Python workspace from Drive without pretending the VM is persistent."""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport site\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport uuid\nfrom datetime import datetime, timezone\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with temporary.open("w") as stream:\n        json.dump(value, stream, indent=2)\n        stream.write("\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temporary, path)\n\n\ndef runtime_signature():\n    # Base packages are inherited, rather than copied into each environment.\n    # A changed base image requires an explicit rebuild, never a silent restore.\n    packages = json.loads(subprocess.check_output(\n        [sys.executable, "-m", "pip", "list", "--format=json"], text=True))\n    normalized = sorted((p["name"].lower().replace("_", "-"), p["version"])\n                        for p in packages)\n    return {\n        "python": list(sys.version_info[:2]),\n        "machine": platform.machine(),\n        "os": platform.system(),\n        "libc": list(platform.libc_ver()),\n        "base_packages_sha256": hashlib.sha256(\n            json.dumps(normalized).encode()).hexdigest(),\n    }\n\n\nclass Workspace:\n    """Use durable files on Drive and executable packages on the current VM."""\n\n    def __init__(self, project="default", drive_root=None, local_root=None):\n        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", project):\n            raise ValueError("Project must use letters, digits, underscores or hyphens.")\n        default_drive = Path("/content/drive/MyDrive")\n        if drive_root is None and not default_drive.is_dir():\n            raise RuntimeError("Mount Google Drive before creating a workspace.")\n        self.project = project\n        # Subprocesses run with cwd=self.root, so relative paths must be absolute.\n        self.root = (Path(drive_root or default_drive / "PersistentColab" / "workspaces") / project).resolve()\n        self.local = (Path(local_root or "/content/persistent-colab") / project).resolve()\n        self.venv = self.local / ".venv"\n        self.python = self.venv / "bin" / "python"\n        self.env_dir = self.root / "environment"\n        self.snapshots = self.env_dir / "snapshots"\n        self.latest = self.env_dir / "latest.json"\n        self.signature = runtime_signature()\n        self.activated = False\n        for folder in ("src", "data", "models", "outputs", "checkpoints", "environment/snapshots"):\n            (self.root / folder).mkdir(parents=True, exist_ok=True)\n        self.local.mkdir(parents=True, exist_ok=True)\n        # Model files are durable, tokens remain in the ephemeral runtime.\n        os.environ["HF_HUB_CACHE"] = str(self.root / "models" / "huggingface")\n        os.environ["HF_TOKEN_PATH"] = str(self.local / "hf-token")\n        os.environ["TORCH_HOME"] = str(self.root / "models" / "torch")\n\n    def prepare(self, restore=True):\n        """Recreate executable paths and restore the latest compatible snapshot."""\n        ready = self.local / "environment-ready.json"\n        if self.python.exists():\n            if ready.exists() and json.loads(ready.read_text()) == self.signature:\n                return self\n            raise RuntimeError("Incomplete local environment. Start a fresh runtime to restore safely.")\n        snapshot = json.loads(self.latest.read_text()) if restore and self.latest.exists() else None\n        if snapshot and snapshot["runtime"] != self.signature:\n            raise RuntimeError(\n                "The Colab Python/base image changed. The saved environment was not loaded. "\n                "Review environment/latest.json, then explicitly run ws.rebuild() to "\n                "reinstall the saved requirements on this image. Old snapshots are kept.")\n        if snapshot and snapshot["local_path"] != str(self.venv):\n            raise RuntimeError("Snapshot executable paths differ. Use the original project name and runtime path.")\n        # Colab\'s image has pip but its ensurepip bootstrap is unavailable.\n        # Inherit pip from the base image; its install target is still this venv.\n        subprocess.run([sys.executable, "-m", "venv", "--without-pip",\n                        "--system-site-packages", str(self.venv)], check=True)\n        if snapshot:\n            archive_name = snapshot["archive"]\n            if Path(archive_name).name != archive_name:\n                raise ValueError("Invalid snapshot archive name.")\n            source = self.snapshots / archive_name\n            local_archive = self.local / "restore.tar.gz"\n            shutil.copyfile(source, local_archive)\n            if sha256(local_archive) != snapshot["sha256"]:\n                shutil.rmtree(self.venv)\n                local_archive.unlink()\n                raise RuntimeError("Snapshot checksum mismatch; environment was not restored.")\n            try:\n                with tarfile.open(local_archive, "r:gz") as archive:\n                    archive.extractall(self.venv, filter="data")\n            except Exception:\n                # This directory is an agent-owned, disposable VM environment only.\n                shutil.rmtree(self.venv)\n                raise\n            finally:\n                local_archive.unlink(missing_ok=True)\n            print("Restored Python environment:", archive_name)\n        else:\n            print("Created a new Python environment; Colab base packages are available.")\n        write_json(ready, self.signature)\n        print("Persistent workspace:", self.root)\n        return self\n\n    def run(self, code, capture=False):\n        """Run Python in the saved environment; persistent workspace is its cwd."""\n        return subprocess.run([str(self.python), "-c", code], cwd=self.root,\n                              check=True, text=True, capture_output=capture)\n\n    def install(self, *packages):\n        """Install into the environment and immediately persist its new snapshot."""\n        if not packages or any(p.startswith("-") for p in packages):\n            raise ValueError("Pass package specifications, such as \'humanize==4.15.0\'.")\n        subprocess.run([str(self.python), "-m", "pip", "install", *packages], check=True)\n        result = self.save_environment()\n        importlib.invalidate_caches()\n        print("New packages are available in fresh imports; restart before replacing loaded packages.")\n        return result\n\n    def save_environment(self):\n        """Commit an immutable archive, publishing the pointer only after readback."""\n        if not self.python.exists():\n            raise RuntimeError("Run ws.prepare() first.")\n        requirements = subprocess.check_output(\n            [str(self.python), "-m", "pip", "freeze", "--local"], text=True)\n        # Do not persist credentials embedded in installation URLs.\n        if "://" in requirements or "-e " in requirements or " @ " in requirements:\n            raise RuntimeError("Use versioned index packages; save local/editable source under ws.root / \'src\'.")\n        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]\n        name = stamp + ".tar.gz"\n        with tempfile.TemporaryDirectory(dir=self.local) as temporary:\n            archive_path = Path(temporary) / name\n            with tarfile.open(archive_path, "w:gz", compresslevel=1) as archive:\n                for directory in ("lib", "bin", "include"):\n                    source = self.venv / directory\n                    if source.exists():\n                        def exclude_interpreter(info):\n                            if info.name.startswith("bin/python"):\n                                return None\n                            if info.issym() and (self.venv / info.name).resolve() == Path(sys.executable).resolve():\n                                return None\n                            if info.issym() and Path(info.linkname).is_absolute():\n                                raise RuntimeError("Environment contains an external symlink: " + info.name)\n                            if "__pycache__" in Path(info.name).parts:\n                                return None\n                            return info\n                        archive.add(source, arcname=directory, filter=exclude_interpreter)\n            expected = sha256(archive_path)\n            destination = self.snapshots / name\n            with archive_path.open("rb") as source, destination.open("xb") as target:\n                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)\n                target.flush()\n                os.fsync(target.fileno())\n            if sha256(destination) != expected:\n                raise RuntimeError("Drive snapshot readback failed; previous snapshot remains current.")\n        snapshot = {"schema": 1, "archive": name, "sha256": expected,\n                    "created_utc": stamp, "runtime": self.signature,\n                    "requirements": requirements, "local_path": str(self.venv)}\n        write_json(self.snapshots / (stamp + ".json"), snapshot)\n        write_json(self.env_dir / "latest.json", snapshot)\n        print("Environment saved and checksum verified:", name)\n        return snapshot\n\n    def rebuild(self):\n        """Explicitly reinstall recorded packages when Colab\'s base image changed."""\n        if self.activated:\n            raise RuntimeError("Restart the kernel before rebuilding an activated environment.")\n        previous = json.loads(self.latest.read_text())\n        if self.venv.exists():\n            raise RuntimeError("Use a fresh runtime for rebuilding; current environment is preserved.")\n        self.prepare(restore=False)\n        lock = self.local / "requirements.txt"\n        lock.write_text(previous["requirements"])\n        if previous["requirements"].strip():\n            subprocess.run([str(self.python), "-m", "pip", "install", "-r", str(lock)], check=True)\n        return self.save_environment()\n\n    def activate(self):\n        """Expose this environment to fresh imports in ordinary notebook cells."""\n        paths = json.loads(subprocess.check_output(\n            [str(self.python), "-c", "import site,json; print(json.dumps(site.getsitepackages()))"], text=True))\n        own = [p for p in paths if Path(p).is_relative_to(self.venv)]\n        for path in reversed(own):\n            site.addsitedir(path)\n            sys.path.remove(path)\n            sys.path.insert(0, path)\n        os.environ["PATH"] = str(self.venv / "bin") + os.pathsep + os.environ["PATH"]\n        self.activated = True\n        importlib.invalidate_caches()\n        os.chdir(self.root)\n        print("Notebook imports use the saved environment. Install through ws.install(...).")\n        return self\n\n    def write_checkpoint(self, state, name="session.json"):\n        """Save explicit JSON state; arbitrary RAM/GPU state is never implicit."""\n        if Path(name).name != name or not name.endswith(".json"):\n            raise ValueError("Checkpoint name must be a simple .json filename.")\n        write_json(self.root / "checkpoints" / name, state)\n        return self.root / "checkpoints" / name\n\n    def read_checkpoint(self, name="session.json"):\n        if Path(name).name != name or not name.endswith(".json"):\n            raise ValueError("Checkpoint name must be a simple .json filename.")\n        path = self.root / "checkpoints" / name\n        return json.loads(path.read_text()) if path.exists() else None\n\n    def flush(self):\n        """Flush Drive writes before the user releases the VM."""\n        from google.colab import drive\n        drive.flush_and_unmount()\n        print("Drive writes flushed. You may now disconnect and delete this runtime.")\n')
ws = Workspace(PROJECT)
ws.prepare()
ws.activate()
print('Previous saved state:', ws.read_checkpoint())


## Work here

Your current directory is the durable Drive workspace. Use `ws.root / 'src'`, `'data'`, `'models'`, `'outputs'`, and `'checkpoints'` for files that must survive.

Install additional packages with `ws.install('package==version')`; it saves the environment immediately. The environment inherits Colab's preinstalled AI libraries. Do not use `%pip` or `!pip` for packages you want this environment to retain. Use `ws.run(code)` or `subprocess.run([str(ws.python), ...])` for an independent Python process and for packages already imported by the notebook kernel. Installing a replacement for an already loaded package requires restarting the kernel and running cells 1 and 2 again.

Hugging Face model downloads and Torch downloads use the Drive model directory. Authentication tokens stay on the temporary runtime. Keep secrets in Colab Secrets; never in the notebook, workspace or checkpoints.

For many small files, unpack a dataset on the runtime's local disk for speed, retaining its original archive on Drive. Only files actually saved back to Drive survive. Concurrent writers to the same project are unsupported.


In [ ]:
# @title Optional: install and save packages
# Uncomment and change the package list when needed.
# ws.install('humanize==4.15.0')


In [ ]:
# @title Your Python / AI code
from pathlib import Path
print('Working directory:', Path.cwd())
print('Persistent output directory:', ws.root / 'outputs')
print('Environment Python:', ws.python)
# Write your code in this cell or add more cells here.


## Save progress explicitly

Use `ws.write_checkpoint({'step': 100, ...})` for JSON state; load it with `ws.read_checkpoint()`. This is application-defined state, not an automatic copy of Python variables.

For Hugging Face Trainer, set `output_dir=str(ws.root / 'checkpoints' / 'training')`, save checkpoints regularly, and use `trainer.train(resume_from_checkpoint=...)` to resume a complete checkpoint. Saving model weights alone does not preserve optimizer, scheduler and training progress. Keep the model, dataset and training configuration compatible when resuming.

System installations (`apt`), OS services and driver changes are not included in the Python snapshot. Put necessary setup commands in explicit notebook cells and rerun them on a new VM.


In [ ]:
# @title Save environment
saved_environment = ws.save_environment()
print('Saved at:', saved_environment['created_utc'])


## Before releasing compute

Check **FLUSH_DRIVE** and run the next cell **only when finished**. It flushes and unmounts Drive. Then disconnect and delete the runtime using the Runtime menu. To continue working after unmounting, mount Drive again.

If Colab reports that its Python/base image changed, restart with a fresh runtime, run cell 1 and instantiate `ws` using cell 2; after the compatibility error, run `ws.rebuild()` and `ws.activate()` in a separate cell. This explicitly reinstalls the recorded package versions and saves a new snapshot; previous snapshots remain on Drive. Reinstallation can fail when a package no longer supports the new image.


In [ ]:
# @title Flush Drive (run manually when finished)
FLUSH_DRIVE = False  # @param {type:"boolean"}
if FLUSH_DRIVE:
    ws.flush()
else:
    print('Check FLUSH_DRIVE and run this cell when you are finished.')
